# Experiment: SpectralBio Final Accept Hardening Part 4 (A100)

This notebook is the benchmark-freezing notebook for BRCA2.

## Scope
- Freeze BRCA2 as a second canonical public benchmark candidate
- Run BRCA2 nested CV with fixed and tuned alpha
- Produce 150M and 650M public artifacts, with optional 3B shadow evidence
- Emit manifests, checksums, score files, verification payloads, and release-ready notes

## Intended runtime
- Target hardware: A100
- Final replay remains CPU-only after artifacts are frozen


In [ ]:
from pathlib import Path

USE_GOOGLE_DRIVE = False
DRIVE_MOUNT_POINT = Path('/content/drive')
DRIVE_OUTPUT_SUBDIR = Path('MyDrive/SpectralBioFinalAcceptA100')

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount(str(DRIVE_MOUNT_POINT))
    print('Drive mounted at', DRIVE_MOUNT_POINT)
else:
    print('Drive mount skipped; outputs stay in the VM unless OUTPUT_ROOT is changed later.')


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/DaviBonetto/SpectralBio.git'
REPO_BRANCH = 'codex/claw4s-rebuild'
REPO_ROOT = Path('/content/Stanford-Claw4s')
BOOTSTRAP_MARKER = REPO_ROOT / '.colab_bootstrap_v3_complete'

if not REPO_ROOT.exists():
    subprocess.check_call(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_ROOT)])

os.chdir(REPO_ROOT)
subprocess.check_call(['git', 'fetch', 'origin', REPO_BRANCH])
subprocess.check_call(['git', 'checkout', REPO_BRANCH])
src_path = REPO_ROOT / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

if not BOOTSTRAP_MARKER.exists():
    subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchvision'])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', '.', '--no-deps'])
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install',
        'numpy==2.1.3', 'plotly==5.24.1', 'pyyaml==6.0.2', 'scikit-learn==1.5.2',
        'scipy==1.14.1', 'transformers==4.49.0', 'pandas', 'matplotlib'
    ])
    BOOTSTRAP_MARKER.write_text('ok\n', encoding='utf-8')
    print('Dependencies installed. Continuing in the same runtime.')
else:
    print('Bootstrap marker found; skipping reinstall.')



## Plan

- Build the second benchmark once and freeze it properly.
- Separate public contract outputs from heavier shadow evidence.
- Make BRCA2 mechanically reproducible before we ask the manuscript to lean on it.


In [ ]:
import subprocess
import sys
import pandas as pd

try:
    from spectralbio.supplementary.final_accept_hardening import (
        AcceptHardeningConfig,
        create_accept_hardening_paths,
        freeze_brca2_canonical_benchmark,
        render_brca2_canonical_release_artifacts,
    )
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', '.', '--no-deps'])
    from spectralbio.supplementary.final_accept_hardening import (
        AcceptHardeningConfig,
        create_accept_hardening_paths,
        freeze_brca2_canonical_benchmark,
        render_brca2_canonical_release_artifacts,
    )

PUBLIC_MODELS = ('facebook/esm2_t30_150M_UR50D', 'facebook/esm2_t33_650M_UR50D')
RUN_3B_SHADOW = True
OVERWRITE = False

OUTPUT_ROOT = (
    DRIVE_MOUNT_POINT / DRIVE_OUTPUT_SUBDIR
    if USE_GOOGLE_DRIVE
    else REPO_ROOT / 'supplementary' / 'final_accept_a100'
)

config = AcceptHardeningConfig(
    stronger_model_names=('facebook/esm2_t33_650M_UR50D', 'facebook/esm2_t36_3B_UR50D'),
    overwrite=OVERWRITE,
)
paths = create_accept_hardening_paths(repo_root=REPO_ROOT, output_root=OUTPUT_ROOT)
print(paths)
print('Public BRCA2 models:', PUBLIC_MODELS)
print('Run 3B shadow:', RUN_3B_SHADOW)



In [ ]:
brca2_release = freeze_brca2_canonical_benchmark(
    paths=paths,
    config=config,
    public_models=PUBLIC_MODELS,
    run_3b_shadow=RUN_3B_SHADOW,
)
release_artifacts = render_brca2_canonical_release_artifacts(paths, brca2_release)
display(pd.DataFrame(brca2_release['model_rows']))
display(pd.DataFrame(brca2_release['nested_cv_rows']).head())
print(release_artifacts)



In [ ]:
expected_paths = [
    paths.root / 'brca2_canonical' / 'brca2_canonical_manifest.json',
    paths.root / 'brca2_canonical' / 'brca2_scores_150m.tsv',
    paths.root / 'brca2_canonical' / 'brca2_scores_650m.tsv',
    paths.root / 'brca2_canonical' / 'brca2_verification.json',
]
for path in expected_paths:
    print(path)


In [ ]:
print('Part 4 scaffold complete. This notebook is reserved for BRCA2 canonicalization on A100.')
